# 02 — Backbone baseline training

This trains a multi-label CNN baseline using BCEWithLogitsLoss and validation macro average precision selection. Use ResNet18 for fast P100 development; switch to EfficientNet-B0/B2 when you want a stronger backbone.

In [ ]:

from pathlib import Path
import sys, os, json, time

# Locate the project root whether this folder is in /kaggle/working or uploaded as a Kaggle dataset.
candidates = [Path.cwd(), Path('/kaggle/working/ladi_v2_gcn_project')]
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').glob('*'))
PROJECT_ROOT = None
for c in candidates:
    if (c / 'src' / 'ladi_config.py').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find project src/. Upload/unzip the full ladi_v2_gcn_project folder first.')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from pathlib import Path
import torch
import pandas as pd

from src.ladi_config import LABEL_COLS_V2A
from src.ladi_data import make_loaders
from src.ladi_models import BaselineCNN, init_classifier_bias, pos_weight_from_df
from src.ladi_train import seed_everything, train_model

seed_everything(42)
CACHE_DIR = Path('/kaggle/working/ladi_v2a_cache')
OUT_DIR = Path('/kaggle/working/ladi_v2_outputs')
RUN_DIR = OUT_DIR / 'runs'
RUN_DIR.mkdir(parents=True, exist_ok=True)

BACKBONE = 'resnet18'      # options: resnet18, resnet34, efficientnet_b0, efficientnet_b2
IMG_SIZE = 320             # P100-safe. Try 384 with batch 8 if memory allows.
BATCH_SIZE = 16
EPOCHS = 8                 # Increase to 15-20 for final experiments.
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
AMP = True


In [ ]:
train_loader, val_loader, test_loader, label_cols, train_df, val_df, test_df = make_loaders(
    CACHE_DIR, label_cols=LABEL_COLS_V2A, img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
train_freq = train_df[label_cols].mean().values
pos_weight = pos_weight_from_df(train_df, label_cols, max_weight=8.0)
print('labels:', label_cols)
print('pos_weight:', pos_weight)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BaselineCNN(num_labels=len(label_cols), backbone=BACKBONE, pretrained=True, dropout=0.25)
init_classifier_bias(model, train_freq)
metrics, ckpt_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    out_dir=RUN_DIR,
    run_name=f'baseline_{BACKBONE}_{IMG_SIZE}',
    label_cols=label_cols,
    train_freq=train_freq,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    pos_weight=pos_weight,
    amp=AMP,
)
print('Best checkpoint:', ckpt_path)
metrics['test_at_0_5']


Next: run **03_static_gcn_training.ipynb** to train a graph classifier using the train-label co-occurrence graph.